ITEM 1:

In [ ]:
from sqlalchemy import create_engine, text
import geopandas as gpd
from psycopg2 import sql
import geoalchemy2

gdf = gpd.read_file("peru_distrital_full.geojson")
engine = create_engine("postgresql://postgres@/lab5?host=/run/postgresql")

with engine.connect() as conn:
    conn.execute(text("CREATE EXTENSION IF NOT EXISTS postgis;"))
    conn.commit()

gdf.to_postgis("peru_distrital_full", engine, if_exists="replace")

with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM peru_distrital_full;"))
    count = result.scalar()
    print(f"Total rows in peru_distrital_full: {count}")

Total rows in peru_distrital_full: 1875


Utilizar PgAdmin para visualizar los datos geoespaciales y adjuntar screenshot

![alt text](image.png)
![alt text](image-1.png)

ITEM 2 \
Vamos a usar la misma ciudad, en este caso Lima, pero efocandonos en el lado Sur. La cual nos da la siguiente area

![alt text](image-2.png)

ITEM 3 \
En esta parte leemos los centros poblados del archivo Shapefile y lo cargamos a la base de datos

In [15]:
gdf_cpp = gpd.read_file("peru_centros_poblados/CCPP_IGN100K.shp")
print(gdf_cpp.head())

   OBJECTID   NOM_POBLAD FUENTE      CÓDIGO CAT_POBLAD      DIST     PROV  \
0         1   PANDISHARI   INEI  2502010002      OTROS  RAYMONDI  ATALAYA   
1         2      CHICOSA   INEI  2502010003      OTROS  RAYMONDI  ATALAYA   
2         3         RAYA    IGN  2502010004      OTROS  RAYMONDI  ATALAYA   
3         4  PENSILVANIA   INEI  2502010005      OTROS  RAYMONDI  ATALAYA   
4         5  PONTE VEDRA   INEI  2502010006    CASERÍO  RAYMONDI  ATALAYA   

       DEP CÓD_INT             CATEGORIA         X         Y     N_BUSQDA  \
0  UCAYALI    2050  Centro Poblado Menor -74.06462 -10.37129   PANDISHARI   
1  UCAYALI    2050  Centro Poblado Menor -74.06153 -10.37852      CHICOSA   
2  UCAYALI    2350  Centro Poblado Menor -72.94118 -10.33043         RAYA   
3  UCAYALI    2050  Centro Poblado Menor -74.05988 -10.40401  PENSILVANIA   
4  UCAYALI    2050  Centro Poblado Menor -74.03788 -10.41809  PONTE VEDRA   

                      geometry  
0  POINT (-74.06462 -10.37129)  
1  POINT

In [20]:
engine = create_engine("postgresql://postgres@/lab5?host=/run/postgresql")
# Creamos el Indice GIST para la columna de geometría
with engine.connect() as conn:
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_ccpp_geometry ON \"CCPP_IGN100K\" USING GIST (geometry);"))
    conn.commit()

gdf_cpp.to_postgis("CCPP_IGN100K", engine, if_exists="replace")


Para este item, vamos a tomar de punto referencial a la ciudad de 'Cajamarca' y vamos a mostrar sus 5 centros poblados más cercanos

![alt text](image-3.png)

![alt text](image-4.png)

ITEM 4 \
Consulta para la resolución de este item:

![alt text](image-5.png)

![alt text](image-6.png)